In [0]:
from pyspark.sql.functions import current_timestamp,col,lit

In [0]:
%sql
create volume if not exists dev.dbx_course.landing

In [0]:
dbutils.fs.mkdirs("/Volumes/dev/dbx_course/landing/orders")
dbutils.fs.mkdirs("/Volumes/dev/dbx_course/landing/customers")

In [0]:
landingpath = "/Volumes/dev/dbx_course/landing"
checkpoint_location = f"{landingpath}/_checkpoints/orders"
schema_location = f"{landingpath}/_schemas/orders"

orders_raw = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format","csv")
         .option("cloudFiles.inferColumnTypes",True)
         .option("cloudFiles.schemaLocation",schema_location)
         .option("header",True)
         .load(f"{landingpath}/orders/")
)

In [0]:
orders_bronze = (
    orders_raw.withColumn("_source",lit("adls_auto_loader"))
              .withColumn("_ingestion_time",current_timestamp())
              .withColumn("_sourcefile",col("_metadata.file_name"))
)

In [0]:
(
    orders_bronze.writeStream
                 .option("checkpointLocation",checkpoint_location)
                 .option("mergeSchema",True)
                 .trigger(availableNow=True)
                 .table("dev.default.orders")
                 .awaitTermination()
)

In [0]:
spark.table("dev.default.orders").display()